In [1]:
import json
import os
import textwrap
import warnings

from dotenv import load_dotenv

warnings.filterwarnings("ignore")



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

OPENAI_ENV = "/Users/shivam13juna/Documents/scaler/GEN_AI_REF/openai_key.env"
loaded = load_dotenv(OPENAI_ENV)
if not loaded:
    raise FileNotFoundError(f"No env file at {OPENAI_ENV}")

api_key = os.getenv("OPENAI_API_KEY")


---
# 📖 Block 1: The Evaluation Problem — Why Traditional Metrics Fail
### ⏱️ ~15 minutes

## The Story

> *Imagine you've built a customer support chatbot for **TechMart**, an online electronics store. It answers 1,000 questions a day. Your boss asks: "How good is it?"*
>
> *You check accuracy — but against what? There's no single right answer to "Can I return a partially used product?" The answer depends on tone, policy nuance, completeness, and empathy.*
>
> *Welcome to the hardest unsolved problem in GenAI engineering.*

Let's see this problem in action with a concrete example.

In [2]:
import truststore
truststore.inject_into_ssl()
from openai import OpenAI
client = OpenAI(api_key=api_key)



# Our scenario: TechMart customer support chatbot
question = "What's your return policy for electronics?"

expected_answer = (
    "You can return most electronics within 30 days of purchase for a full refund. "
    "Items must be in original packaging with all accessories included. "
    "Opened software and digital downloads are non-refundable. "
    "For defective items, we offer a 90-day exchange warranty."
)

# Let's generate a chatbot response using OpenAI
response = client.responses.create(
    model="gpt-5-nano",
    input=[
        {"role": "system", "content": (
            "You are a helpful customer support agent for TechMart electronics store. "
            "TechMart's return policy: 30-day returns for electronics in original packaging "
            "with accessories. Opened software/digital downloads non-refundable. "
            "90-day exchange warranty for defective items."
        )},
        {"role": "user", "content": question}
    ]
)

actual_answer = response.output_text
print("📋 QUESTION:", question)
print()
print("✅ EXPECTED ANSWER:")
print(expected_answer)
print()
print("🤖 CHATBOT ANSWER:")
print(actual_answer)

📋 QUESTION: What's your return policy for electronics?

✅ EXPECTED ANSWER:
You can return most electronics within 30 days of purchase for a full refund. Items must be in original packaging with all accessories included. Opened software and digital downloads are non-refundable. For defective items, we offer a 90-day exchange warranty.

🤖 CHATBOT ANSWER:
Here’s our electronics return policy:

- 30-day returns: Electronics can be returned within 30 days as long as they are in their original packaging with all accessories.
- Opened software/digital downloads: Non-refundable.
- Defective items: We offer a 90-day exchange warranty for items that are defective.

If you’d like, tell me the item and purchase date and I can confirm eligibility or help with the next steps.


In [3]:
misleading_answer = (
    "You can return most electronics within 30 days of purchase for a full refund. "
    "Items must be in original packaging with all accessories included. "
    "We also offer FREE LIFETIME WARRANTY on everything and PRICE MATCHING "
    "against any competitor!"  
)

print("🤖 A SECOND ANSWER — same question, mostly right, quietly invented:")
print(misleading_answer)
print()
print("The first two sentences are policy. The last one is fabricated:")
print("TechMart has no lifetime warranty and no price matching.")
print("Hold on to this one — every metric in this notebook gets judged on it.")


🤖 A SECOND ANSWER — same question, mostly right, quietly invented:
You can return most electronics within 30 days of purchase for a full refund. Items must be in original packaging with all accessories included. We also offer FREE LIFETIME WARRANTY on everything and PRICE MATCHING against any competitor!

The first two sentences are policy. The last one is fabricated:
TechMart has no lifetime warranty and no price matching.
Hold on to this one — every metric in this notebook gets judged on it.


## The Four Pillars of GenAI Evaluation

| Approach | How It Works | Best For | Limitation |
|----------|-------------|----------|------------|
| **Heuristic / Code-Based** | Regex, format checks, length constraints | Structured outputs (JSON), format compliance | Can't judge meaning |
| **Statistical / NLP** | BLEU, ROUGE, BERTScore, cosine similarity | Translation, summarization (with reference) | Poor correlation with human judgment |
| **LLM-as-a-Judge** | A powerful LLM scores output against a rubric | Open-ended quality, custom criteria, scale | Judge can be biased; needs calibration |
| **Human Evaluation** | Domain experts rate on defined criteria | High-stakes, ground truth calibration | Expensive, slow, subjective |

### In production, teams use ALL FOUR in layers:

```
Layer 5: Continuous observability (Langfuse)           ← always watching
Layer 4: Human evaluation for calibration              ← monthly
Layer 3: Statistical metrics for regression tracking   ← weekly
Layer 2: LLM-as-a-Judge on key dimensions             ← every deploy
Layer 1: Code-based checks in CI/CD                   ← every commit
```

In [4]:

# Pillar 1: Heuristic / Code-Based Evaluation

# These are simple but catch real problems in production!

print("Actual answer: ", actual_answer)

def evaluate_heuristics(response: str) -> dict:
    """Basic code-based checks for a customer support chatbot."""
    checks = {}

    # Length check — too short = probably unhelpful, too long = overwhelming
    word_count = len(response.split())
    checks["appropriate_length"] = 20 <= word_count <= 300
    checks["word_count"] = word_count

    # Contains required elements
    checks["has_greeting_or_direct_answer"] = not response.startswith("I don't")
    checks["no_competitor_mentions"] = not any(
        comp in response.lower() for comp in ["amazon", "bestbuy", "best buy", "walmart"]
    )

    # Safety checks
    checks["no_profanity"] = not any(
        word in response.lower() for word in ["damn", "hell", "stupid"]
    )

    # Format check — should not contain raw code or system prompts
    checks["no_system_prompt_leak"] = "system:" not in response.lower()
    checks["no_raw_json"] = not response.strip().startswith("{")

    return checks

# Test on our chatbot's response
print("🔍 HEURISTIC CHECKS ON CHATBOT RESPONSE:")
print("=" * 50)
results = evaluate_heuristics(actual_answer)
for check, passed in results.items():
    status = "✅" if (passed if isinstance(passed, bool) else True) else "❌"
    print(f"  {status} {check}: {passed}")

print()
print("💡 These checks are fast and deterministic — perfect for CI/CD.")
print("   But they can't tell you if the answer is ACTUALLY CORRECT or HELPFUL.")

Actual answer:  Here’s our electronics return policy:

- 30-day returns: Electronics can be returned within 30 days as long as they are in their original packaging with all accessories.
- Opened software/digital downloads: Non-refundable.
- Defective items: We offer a 90-day exchange warranty for items that are defective.

If you’d like, tell me the item and purchase date and I can confirm eligibility or help with the next steps.
🔍 HEURISTIC CHECKS ON CHATBOT RESPONSE:
  ✅ appropriate_length: True
  ✅ word_count: 67
  ✅ has_greeting_or_direct_answer: True
  ✅ no_competitor_mentions: True
  ✅ no_profanity: True
  ✅ no_system_prompt_leak: True
  ✅ no_raw_json: True

💡 These checks are fast and deterministic — perfect for CI/CD.
   But they can't tell you if the answer is ACTUALLY CORRECT or HELPFUL.


In [5]:
print("🔍 HEURISTIC CHECKS ON CHATBOT RESPONSE:")
print("=" * 50)
results = evaluate_heuristics(misleading_answer)
for check, passed in results.items():
    status = "✅" if (passed if isinstance(passed, bool) else True) else "❌"
    print(f"  {status} {check}: {passed}")

🔍 HEURISTIC CHECKS ON CHATBOT RESPONSE:
  ✅ appropriate_length: True
  ✅ word_count: 38
  ✅ has_greeting_or_direct_answer: True
  ✅ no_competitor_mentions: True
  ✅ no_profanity: True
  ✅ no_system_prompt_leak: True
  ✅ no_raw_json: True


In [6]:

# Pillar 2: LLM-as-a-Judge — Build one from scratch!

# Before we use frameworks, let's understand what's happening under the hood.

def llm_judge(question, response, criteria, model="gpt-5-nano"):
    """A simple LLM-as-a-Judge implementation from scratch."""

    judge_prompt = f"""You are an expert evaluator for a customer support chatbot.

    Evaluate the following response on this criteria: {criteria}

    USER QUESTION: {question}
    CHATBOT RESPONSE: {response}

    Score from 1-5 where:
    1 = Completely fails the criteria
    2 = Mostly fails with minor positives
    3 = Partially meets criteria
    4 = Mostly meets criteria with minor issues
    5 = Fully meets criteria

    Respond in this exact JSON format:
    {{"score": <int>, "reason": "<brief explanation>"}}"""

    result = client.responses.create(
        model=model,
        input=[{"role": "user", "content": judge_prompt}]
    )

    try:
        # Parse JSON from response
        text = result.output_text.strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text)
    except:
        return {"score": 0, "reason": f"Failed to parse: {result.output_text[:200]}"}

# ----- Evaluate on multiple criteria -----
criteria_list = {
    "Accuracy": "Is the response factually correct based on TechMart's return policy?",
    "Helpfulness": "Does the response fully address the user's question in a helpful way?",
    "Tone": "Is the tone professional, friendly, and empathetic?",
    "Completeness": "Does the response cover all relevant aspects (timeframe, conditions, exceptions)?"
}

print("🧑‍⚖️ LLM-AS-A-JUDGE EVALUATION")
print("=" * 60)
print(f"Question: {question}")
print(f"Response: {actual_answer[:150]}...")
print()

scores = {}
for name, criteria in criteria_list.items():
    result = llm_judge(question, actual_answer, criteria)
    scores[name] = result
    print(f"  {name}: {'⭐' * result['score']}{'☆' * (5-result['score'])} ({result['score']}/5)")
    print(f"    → {result['reason']}")
    print()

avg_score = sum(s['score'] for s in scores.values()) / len(scores)
print(f"📊 Average Score: {avg_score:.1f}/5")

🧑‍⚖️ LLM-AS-A-JUDGE EVALUATION
Question: What's your return policy for electronics?
Response: Here’s our electronics return policy:

- 30-day returns: Electronics can be returned within 30 days as long as they are in their original packaging wi...

  Accuracy: ⭐☆☆☆☆ (1/5)
    → Insufficient information to verify against TechMart's official return policy; cannot confirm whether the stated terms (30-day electronics returns, non-refundable opened software/digital downloads, 90-day exchange for defective items) match TechMart's actual policy.

  Helpfulness: ⭐⭐⭐☆☆ (3/5)
    → The answer covers the basic electronics return window and defective-item warranty, but lacks details on refunds/processing, who pays return shipping, and eligibility steps; also includes an irrelevant note about software.

  Tone: ⭐⭐⭐⭐☆ (4/5)
    → Professional and helpful with clear policy; neutral but not very warm or empathetic. A brief empathetic line could strengthen friendliness.

  Completeness: ⭐⭐⭐⭐☆ (4/5)
   

In [7]:
# ----- Now judge the HALLUCINATED response -----
print("🚨 JUDGING THE HALLUCINATED RESPONSE")
print("=" * 60)
print(f"Response: {misleading_answer[:150]}...")
print()

scores = {}
for name, criteria in criteria_list.items():
    result = llm_judge(question, misleading_answer, criteria)
    scores[name] = result
    print(f"  {name}: {'⭐' * result['score']}{'☆' * (5-result['score'])} ({result['score']}/5)")
    print(f"    → {result['reason']}")
    print()

avg_score = sum(s['score'] for s in scores.values()) / len(scores)
print(f"📊 Average Score: {avg_score:.1f}/5")


print("💡 KEY INSIGHT: The LLM judge CATCHES the hallucination that ROUGE missed!")
print("   This is why LLM-as-a-Judge is the dominant evaluation approach in 2025.")

🚨 JUDGING THE HALLUCINATED RESPONSE
Response: You can return most electronics within 30 days of purchase for a full refund. Items must be in original packaging with all accessories included. We al...

  Accuracy: ⭐⭐☆☆☆ (2/5)
    → The reply asserts a 'FREE LIFETIME WARRANTY on everything' and universal price matching, which are highly unlikely to be accurate without qualifiers. It also lacks common policy details (exclusions, restocking fees, eligible electronics, condition). The 30-day full refund and packaging requirement may be plausible, but overall the response cannot be verified against TechMart's official return policy.

  Helpfulness: ⭐⭐⭐☆☆ (3/5)
    → The answer covers the basic 30‑day return window and packaging/accessory requirements, but omits key details (exceptions, how to initiate a return, whether shipping is refunded, restocking fees, and common exclusions). It also adds unrelated offers (lifetime warranty, price matching) that weren’t asked and could confuse.

  Tone: